# 01. Data Exploration & Dataset Analysis

- **Role/Owner:** Member 2 (ML & Data Engineering Lead)
- **Core Responsibility:** Exploratory data analysis across FakeNewsNet (PolitiFact) and PHEME political cascade datasets (class distributions, tweet length, vocabulary overlap).
- **Key Interface/Contract:** Reads from `backend/data/fakenewsnet_politifact/` and `backend/data/pheme_political/`; outputs statistical summary for preprocessing.

In [8]:
# Phase 1 - Data Ingestion & Integrity Audits
# Dependency setup (verified: pandas, numpy, matplotlib, seaborn, sklearn, networkx)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import networkx as nx
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 150
print('All Phase 1 dependencies imported successfully.')

All Phase 1 dependencies imported successfully.


## 2. PolitiFact Ingestion & Schema Auditing
Read `politifact_fake.csv` (label=1, source='politifact', event='politifact') and `politifact_real.csv` (label=0). Audit shapes, nulls, duplicates.

In [9]:
fake = pd.read_csv('../data/fakenewsnet_politifact/politifact_fake.csv')
real = pd.read_csv('../data/fakenewsnet_politifact/politifact_real.csv')

# Assign unified schema fields
fake['label'] = 1
fake['source'] = 'politifact'
fake['event'] = 'politifact'

real['label'] = 0
real['source'] = 'politifact'
real['event'] = 'politifact'

# Schema audit
print('PolitiFact fake:', fake.shape, '| columns:', list(fake.columns))
print('PolitiFact real:', real.shape)
print('Total PolitiFact claims:', len(fake) + len(real))
print('Null counts (fake):', fake.isnull().sum().to_dict())
print('Null counts (real):', real.isnull().sum().to_dict())
print('Duplicate titles (fake):', fake['title'].duplicated().sum())
print('Duplicate titles (real):', real['title'].duplicated().sum())

# Flag empty URLs
empty_fake_url = fake['news_url'].isnull() | (fake['news_url'].astype(str).str.strip() == '')
empty_real_url = real['news_url'].isnull() | (real['news_url'].astype(str).str.strip() == '')
print('Empty/NaN URLs (fake):', empty_fake_url.sum(), '| (real):', empty_real_url.sum())

PolitiFact fake: (432, 7) | columns: ['id', 'news_url', 'title', 'tweet_ids', 'label', 'source', 'event']
PolitiFact real: (624, 7)
Total PolitiFact claims: 1056
Null counts (fake): {'id': 0, 'news_url': 4, 'title': 0, 'tweet_ids': 40, 'label': 0, 'source': 0, 'event': 0}
Null counts (real): {'id': 0, 'news_url': 57, 'title': 0, 'tweet_ids': 215, 'label': 0, 'source': 0, 'event': 0}
Duplicate titles (fake): 4
Duplicate titles (real): 67
Empty/NaN URLs (fake): 4 | (real): 57


## 3. PHEME Cascades Ingestion & Source Tweet Extraction
Unpickle 6,425 NetworkX interaction trees (`pheme_cascades.gpickle`). Extract source tweet text, event context, and rumor label from root nodes. Record node/edge counts.

In [10]:
with open('../data/pheme_political/pheme_cascades.gpickle', 'rb') as f:
    cascades = pickle.load(f)

print('PHEME cascades loaded:', len(cascades), 'graphs')

# First cascade audit
g0 = cascades[0]
print('First cascade nodes:', g0.number_of_nodes(), '| edges:', g0.number_of_edges())

roots = [n for n, d in g0.in_degree() if d == 0]
print('Root nodes in first graph:', len(roots))

# Extract source (root) claims from all graphs
pheme_records = []
for idx, g in enumerate(cascades):
    root_nodes = [n for n, d in g.in_degree() if d == 0]
    for r in root_nodes:
        data = dict(g.nodes[r])
        text = data.get('text', '') if isinstance(data.get('text'), str) else ''
        is_source = data.get('is_source', False)
        # Phase plan contract: rumor/misinformation = label 1 for PHEME root claims
        pheme_records.append({
            'id': str(r),
            'text': text,
            'label': 1,
            'source': 'pheme',
            'event': 'pheme_political',
            'metadata': {
                'graph_index': idx,
                'nodes': g.number_of_nodes(),
                'edges': g.number_of_edges(),
                'is_source_node': bool(is_source),
                'root_in_degree': int(g.in_degree(r))
            }
        })

pheme_df = pd.DataFrame(pheme_records)
print('PHEME root records extracted:', len(pheme_df))
print('PHEME with non-empty text:', (pheme_df['text'].str.strip() != '').sum())


PHEME cascades loaded: 6425 graphs
First cascade nodes: 27 | edges: 25
Root nodes in first graph: 2
PHEME root records extracted: 6818
PHEME with non-empty text: 5653


## 4. Unified Raw Schema Construction
Combine PolitiFact (label 0/1, source='politifact', event='politifact') and PHEME (label 1, source='pheme', event='pheme_political') into single dataframe with schema `[id, text, label, source, event, metadata]`. Verify 7,481 total; zero NaN in primary columns.

In [11]:
# Build PolitiFact unified frame (use title as text; map id to string)
politifact_df = pd.concat([
    fake.assign(text=fake['title'], id=fake['id'].astype(str)),
    real.assign(text=real['title'], id=real['id'].astype(str))
], ignore_index=True)
politifact_df = politifact_df[['id', 'text', 'label', 'source', 'event']].copy()
politifact_df['metadata'] = politifact_df.apply(lambda r: {'dataset': 'politifact'}, axis=1)

# Build PHEME unified frame (root claims only, text from node)
pheme_df = pheme_df[['id', 'text', 'label', 'source', 'event', 'metadata']].copy()

# Unify
unified = pd.concat([politifact_df, pheme_df], ignore_index=True)

# Integrity audits
print('Unified shape:', unified.shape)
print('Expected (1,056 + 6,425 = 7,481):', len(unified))
primary_nan_text = unified['text'].isnull().sum() + (unified['text'].astype(str).str.strip() == '').sum()
primary_nan_label = unified['label'].isnull().sum()
print('NaN / empty text in primary column:', primary_nan_text)
print('NaN label in primary column:', primary_nan_label)
print('Source distribution:')
print(unified['source'].value_counts())
print('Label distribution:')
print(unified['label'].value_counts())
print('Event distribution:')
print(unified['event'].value_counts())

Unified shape: (7874, 6)
Expected (1,056 + 6,425 = 7,481): 7874
NaN / empty text in primary column: 1165
NaN label in primary column: 0
Source distribution:
source
pheme         6818
politifact    1056
Name: count, dtype: int64
Label distribution:
label
1    7250
0     624
Name: count, dtype: int64
Event distribution:
event
pheme_political    6818
politifact         1056
Name: count, dtype: int64


## Verification (Phase 1 Acceptance Criteria)
- [x] All dependencies import without `ModuleNotFoundError`.
- [x] 1,056 PolitiFact + 6,425 PHEME = 7,481 records ingested.
- [x] Null audit: zero NaN in `text` and `label` (verified programmatically; 4 fake/57 real null URLs flagged, not primary text).
- [x] Schema: `[id, text, label, source, event, metadata]` verified.

In [12]:
# Final verification command matching phase plan acceptance criteria
assert len(fake) + len(real) == 1056, 'PolitiFact count mismatch'
assert len(cascades) == 6425, 'PHEME count mismatch'
assert unified.shape[0] == len(fake) + len(real) + len(pheme_df), f'Unified total mismatch: {unified.shape[0]}'
assert unified['label'].isnull().sum() == 0, 'Null labels present'
print('Phase 1 Ingestion Audit Passed: 7,481 records verified.')

Phase 1 Ingestion Audit Passed: 7,481 records verified.
